# SNU AI Token-Level Cross-Attention Final Training and Submission

This experiment caches ConvNeXt spatial maps and unpooled Qwen image/text token states, performs text-to-image cross-attention before pooling, replaces group-mean alignment with cross-attention, feeds fused frame vectors to intra-order scoring, and compares identical five-fold runs with and without the final sequence Transformer.

## 1. Paths and experiment parameters

`QWEN_LAYER_INDICES=(-1,)` uses only the final decoder state. Intermediate states are supported by selecting additional indices, but must use a separate cache/result root because their storage and fingerprint differ.

In [1]:
import os
from pathlib import Path

# Must be set before the first torch import; reduces CUDA fragmentation in Colab.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DRIVE_ROOT = Path(os.environ.get('SNU_DRIVE_ROOT', '/content/drive/MyDrive/snu ai'))
BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_full_train_bundle.tar'
TEST_BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_test_bundle.tar'
CODE_BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_token_fusion_code.tar'
WORKDIR = Path(os.environ.get('SNU_WORKDIR', '/content/snu_ai'))
LOCAL_TOKEN_CACHE = WORKDIR / 'feature_cache/token_fusion_final_layer_v1'
LOCAL_TEST_TOKEN_CACHE = WORKDIR / 'feature_cache/token_fusion_final_layer_v1_test'
DRIVE_TOKEN_SHARDS = DRIVE_ROOT / 'cache/token_fusion_final_layer_v1_shards'
DRIVE_TEST_TOKEN_SHARDS = DRIVE_ROOT / 'cache/token_fusion_final_layer_v1_test_shards'
DRIVE_RESULTS_ROOT = DRIVE_ROOT / 'results/token_fusion_cross_attention_v1'
FINAL_OUTPUT_DIR = DRIVE_RESULTS_ROOT / 'final_scorer'
TEST_CHECKPOINT_PATH = FINAL_OUTPUT_DIR / 'final.pt'
SUBMISSION_PATH = DRIVE_RESULTS_ROOT / 'submission_transformer_full_train.csv'

QWEN_LAYER_INDICES = (-1,)
FEATURE_BATCH_CANDIDATES = (64, 48, 32, 24, 16, 8)
IMAGE_WORKERS = 16
CACHE_SHARD_SIZE = 100
RESIDENT_STORAGE = 'int8'  # Reserve the 80 GB runtime primarily for larger compute batches
HIDDEN_DIM = 256
NUM_HEADS = 4
RESAMPLER_TOKENS = 8
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
VERIFIERS = ('transformer', 'none')
FINAL_VERIFIER = 'transformer'
TRAIN_FINAL_SCORER = True


## 2. Mount Drive, install dependencies, and extract code/data

In [2]:
import subprocess
import shutil
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.13.0', 'accelerate==1.14.0', 'pillow>=10'], check=True)
for path in (DRIVE_TOKEN_SHARDS, DRIVE_TEST_TOKEN_SHARDS, DRIVE_RESULTS_ROOT, LOCAL_TOKEN_CACHE, LOCAL_TEST_TOKEN_CACHE):
    path.mkdir(parents=True, exist_ok=True)
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(CODE_BUNDLE_PATH)
if not TEST_BUNDLE_PATH.is_file():
    raise FileNotFoundError(TEST_BUNDLE_PATH)
WORKDIR.mkdir(parents=True, exist_ok=True)
if not (WORKDIR / 'snuaichallenge_data/test.csv').is_file():
    subprocess.run(['tar', '-xf', str(TEST_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))


Mounted at /content/drive


## 3. Inspect and explicitly confirm the training GPU

Run the first cell, inspect its output, then copy the exact GPU name and rounded VRAM value into the confirmation cell. Training is deliberately blocked until this is done.

In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime.')
DETECTED_GPU_NAME = torch.cuda.get_device_name(0)
DETECTED_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('GPU:', DETECTED_GPU_NAME)
print(f'VRAM: {DETECTED_VRAM_GB:.2f} GiB')
print('Capability:', torch.cuda.get_device_capability(0))


GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.25 GiB
Capability: (8, 0)


In [11]:
# Fill these only after checking the preceding cell. Example: 'NVIDIA A100-SXM4-40GB', 39.56
CONFIRMED_GPU_NAME = 'NVIDIA A100-SXM4-80GB'
CONFIRMED_VRAM_GB = 79.15

if CONFIRMED_GPU_NAME != DETECTED_GPU_NAME:
    raise RuntimeError(f'GPU confirmation mismatch: detected={DETECTED_GPU_NAME!r}, confirmed={CONFIRMED_GPU_NAME!r}')
if CONFIRMED_VRAM_GB is None or abs(CONFIRMED_VRAM_GB - DETECTED_VRAM_GB) > 0.5:
    raise RuntimeError(f'VRAM confirmation mismatch: detected={DETECTED_VRAM_GB:.2f} GiB, confirmed={CONFIRMED_VRAM_GB!r}')
print('GPU configuration confirmed; extraction/training cells are enabled.')


GPU configuration confirmed; extraction/training cells are enabled.


## 4. Load and validate the complete 24-class five-fold data

In [12]:
from collections import Counter

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(BUNDLE_PATH)
if not (WORKDIR / 'snuaichallenge_data/train.csv').is_file():
    subprocess.run(['tar', '-xf', str(BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
# The large train bundle contains an older code snapshot. Always reapply the
# current code overlay after data extraction, then invalidate cached imports.
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))
from snu_ordering.data import load_samples

IMAGE_ROOT = WORKDIR / 'snuaichallenge_data/train'
FULL_FOLDS_CSV = WORKDIR / 'snuaichallenge_data/splits/full_folds_5.csv'
samples = load_samples(FULL_FOLDS_CSV, IMAGE_ROOT, require_labels=True, validate_images=True)
answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
assert len(samples) == 9535 and len(answer_counts) == 24
assert sum(sample.no_ordering is True for sample in samples) == 1478
assert all(1905 <= fold_counts[index] <= 1909 for index in range(5))
print('samples/classes/folds:', len(samples), len(answer_counts), dict(sorted(fold_counts.items())))


samples/classes/folds: 9535 24 {0: 1907, 1: 1907, 2: 1909, 3: 1907, 4: 1905}


## 5. Restore resumable unpooled-token cache shards

In [13]:
import tarfile

persisted_relative_paths = set()
cache_shards = sorted(DRIVE_TOKEN_SHARDS.glob('token_cache_*.tar'))
for shard in cache_shards:
    with tarfile.open(shard, 'r') as archive:
        members = [member for member in archive.getmembers() if member.isfile()]
        persisted_relative_paths.update(member.name for member in members)
        archive.extractall(LOCAL_TOKEN_CACHE, members=members, filter='data')
print('restored shards/files:', len(cache_shards), len(persisted_relative_paths))


restored shards/files: 96 9535


## 6. Extract ConvNeXt maps and unpooled Qwen tokens once

This is a new cache. The old pooled `full_train_9535` cache cannot satisfy token-level fusion. Completed files are persisted every 100 samples.

In [20]:
import gc
import shutil

from snu_ordering.features import ConvNeXtTinyFeatureExtractor
from snu_ordering.qwen_representations import FrozenQwenRepresentationExtractor
from snu_ordering.token_fusion_representations import (
    cache_token_representations_batched,
    find_pending_token_representations,
)

pending, valid_paths = find_pending_token_representations(
    samples, LOCAL_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
print('valid/pending:', len(valid_paths), len(pending))
next_shard_index = max([int(path.stem.split('_')[-1]) for path in cache_shards], default=-1) + 1
shard_buffer = []

def persist_saved(paths):
    global next_shard_index, shard_buffer
    shard_buffer.extend(paths)
    while len(shard_buffer) >= CACHE_SHARD_SIZE:
        selected, shard_buffer = shard_buffer[:CACHE_SHARD_SIZE], shard_buffer[CACHE_SHARD_SIZE:]
        local_tar = WORKDIR / f'token_cache_{next_shard_index:04d}.tar'
        final_tar = DRIVE_TOKEN_SHARDS / local_tar.name
        temporary = final_tar.with_suffix('.tar.tmp')
        with tarfile.open(local_tar, 'w') as archive:
            for path in selected:
                archive.add(path, arcname=path.relative_to(LOCAL_TOKEN_CACHE).as_posix())
        shutil.copy2(local_tar, temporary)
        temporary.replace(final_tar)
        local_tar.unlink()
        print('persisted:', final_tar.name)
        next_shard_index += 1

unpersisted_valid = [path for path in valid_paths if path.relative_to(LOCAL_TOKEN_CACHE).as_posix() not in persisted_relative_paths]
persist_saved(unpersisted_valid)

if pending:
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    convnext = ConvNeXtTinyFeatureExtractor(device='cuda')
    qwen = FrozenQwenRepresentationExtractor(device_map='auto', dtype=dtype)
    remaining = pending
    for feature_batch_size in FEATURE_BATCH_CANDIDATES:
        if not remaining:
            break
        try:
            summary = cache_token_representations_batched(
                remaining, convnext=convnext, qwen=qwen,
                qwen_layer_indices=QWEN_LAYER_INDICES, batch_size=feature_batch_size,
                image_workers=IMAGE_WORKERS, on_batch_saved=persist_saved,
            )
            print('feature batch selected:', feature_batch_size, summary)
            remaining = []
        except torch.cuda.OutOfMemoryError:
            print('feature batch OOM; retrying smaller:', feature_batch_size)
            gc.collect()
            torch.cuda.empty_cache()
            remaining, _ = find_pending_token_representations(
                samples, LOCAL_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
            )
    if remaining:
        raise RuntimeError('No safe feature extraction batch was found')
    del qwen, convnext
    gc.collect()
    torch.cuda.empty_cache()

if shard_buffer:
    selected, shard_buffer = shard_buffer, []
    local_tar = WORKDIR / f'token_cache_{next_shard_index:04d}.tar'
    final_tar = DRIVE_TOKEN_SHARDS / local_tar.name
    with tarfile.open(local_tar, 'w') as archive:
        for path in selected:
            archive.add(path, arcname=path.relative_to(LOCAL_TOKEN_CACHE).as_posix())
    temporary = final_tar.with_suffix('.tar.tmp')
    shutil.copy2(local_tar, temporary)
    temporary.replace(final_tar)
    local_tar.unlink()

pending_after, valid_after = find_pending_token_representations(
    samples, LOCAL_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
assert not pending_after and len(valid_after) == 9535


valid/pending: 9535 0


## 7. Build the GPU-resident token store and compiled candidate plans

`auto` selects BF16 only when the padded store consumes at most 55% of VRAM; otherwise it uses per-token symmetric int8 storage on GPU and dequantizes only the active batch. It refuses to fall back to per-sample disk loading.

In [21]:
import importlib
import shutil
import subprocess
import sys

# Section 7 is self-healing: the large data bundle may have restored old code.
subprocess.run(['tar', '-xf', str(CODE_BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
for cache_dir in (WORKDIR / 'snu_ordering').rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for name in list(sys.modules):
    if name == 'snu_ordering' or name.startswith('snu_ordering.'):
        del sys.modules[name]
sys.path.insert(0, str(WORKDIR))
importlib.invalidate_caches()
from snu_ordering.structured_scorer import prepare_candidate_plan
print('section 7 code overlay verified:', prepare_candidate_plan)

from snu_ordering.token_fusion_training import (
    TokenFusionTrainingConfig,
    prepare_resident_token_fusion_samples,
)

DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')
resident_store, prepared_samples = prepare_resident_token_fusion_samples(
    samples, LOCAL_TOKEN_CACHE, DEVICE, top_h=TOP_H, storage_mode=RESIDENT_STORAGE
)
base_config = TokenFusionTrainingConfig(
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    batch_size=128, evaluation_batch_size=256, max_epochs=MAX_EPOCHS,
    patience=PATIENCE, seed=SEED, top_h=TOP_H, top_m=TOP_M,
    alpha=ALPHA, beta=BETA, gamma=GAMMA, delta=DELTA, use_amp=True,
)


ImportError: cannot import name 'prepare_candidate_plan' from 'snu_ordering.structured_scorer' (/content/snu_ai/snu_ordering/structured_scorer.py)

## 8. Probe maximum safe train/evaluation batches with the larger Transformer variant

In [ ]:
from dataclasses import replace
from snu_ordering.token_fusion_scorer import TokenFusionStructuredScorer
from snu_ordering.token_fusion_training import (
    probe_token_fusion_batch_size,
    probe_token_fusion_evaluation_batch_size,
)

def make_scorer(verifier):
    return TokenFusionStructuredScorer(
        qwen_layer_indices=resident_store.qwen_layer_indices,
        hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
        resampler_tokens=RESAMPLER_TOKENS, verifier=verifier,
    ).to(DEVICE)

def plan_work(item):
    plan = item.candidate_plan
    return (
        4 * int(plan.intra_order_indices.shape[0])
        + int(plan.alignment_group_indices.shape[0]),
        plan.latent_count,
    )

probe_scorer = make_scorer('transformer')
fold0_train = [item for item in prepared_samples if item.sample.fold != 0]
worst_case_rows = sorted(fold0_train, key=plan_work, reverse=True)
micro_batch_size = probe_token_fusion_batch_size(
    probe_scorer, resident_store, worst_case_rows, base_config,
    candidates=(96, 64, 48, 32, 24, 16, 8), max_vram_fraction=0.85,
)
evaluation_batch_size = probe_token_fusion_evaluation_batch_size(
    probe_scorer, resident_store, worst_case_rows, base_config,
    candidates=(512, 384, 256, 192, 128, 96, 64, 48, 32), max_vram_fraction=0.85,
)
training_config = replace(
    base_config, micro_batch_size=micro_batch_size,
    evaluation_batch_size=evaluation_batch_size,
)
print(
    'selected logical/micro/eval batch:', training_config.batch_size,
    micro_batch_size, evaluation_batch_size,
)
del probe_scorer
gc.collect()
torch.cuda.empty_cache()


batch probe: batch=96, peak=44.14 GiB, fraction=0.557
evaluation probe: batch=512, OOM
evaluation probe: batch=384, peak=69.86 GiB, fraction=0.882
evaluation probe: batch=256, peak=55.59 GiB, fraction=0.701
selected logical/micro/eval batch: 128 96 256


## 9. Run matched five-fold experiments: Transformer versus no Transformer

Both variants use the same token cache, resident storage, folds, seed, logical train batch, probed micro-batch, evaluation batch, optimizer, and early-stopping policy.

In [ ]:
import random

from snu_ordering.token_fusion_training import train_token_fusion_scorer

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

for verifier in VERIFIERS:
    for fold in FOLDS_TO_RUN:
        output_dir = DRIVE_RESULTS_ROOT / verifier / f'fold_{fold}'
        if (output_dir / 'training_summary.json').is_file():
            print('already completed:', verifier, fold)
            continue
        train_rows = [item for item in prepared_samples if item.sample.fold != fold]
        validation_rows = [item for item in prepared_samples if item.sample.fold == fold]
        seed_everything(SEED)
        scorer = make_scorer(verifier)
        print(f'start {verifier} fold={fold} train={len(train_rows)} val={len(validation_rows)}')
        train_token_fusion_scorer(
            scorer, resident_store, train_rows, validation_rows, output_dir, training_config
        )
        del scorer
        gc.collect()
        torch.cuda.empty_cache()


start transformer fold=0 train=7628 val=1907
{"epoch": 1, "train_loss": 2.8970516364363146, "samples": 1907, "exact_match_accuracy": 0.17461982170949134, "lightweight_top8_recall": 0.6916622968012586, "effective_evaluation_batch_size": 256}
{"epoch": 2, "train_loss": 2.728173601358551, "samples": 1907, "exact_match_accuracy": 0.21027792343995805, "lightweight_top8_recall": 0.6958573675930781, "effective_evaluation_batch_size": 256}
{"epoch": 3, "train_loss": 2.525552253418089, "samples": 1907, "exact_match_accuracy": 0.24016780283167277, "lightweight_top8_recall": 0.702674357629785, "effective_evaluation_batch_size": 256}
{"epoch": 4, "train_loss": 2.3001756521814083, "samples": 1907, "exact_match_accuracy": 0.27215521761929734, "lightweight_top8_recall": 0.7121132669113791, "effective_evaluation_batch_size": 256}
{"epoch": 5, "train_loss": 2.0884620904547417, "samples": 1907, "exact_match_accuracy": 0.29260618772941793, "lightweight_top8_recall": 0.7110644992134242, "effective_evaluat

## 10. Aggregate and compare the two controlled runs

In [ ]:
import json
import statistics

comparison = {}
for verifier in VERIFIERS:
    fold_records = []
    for fold in range(5):
        path = DRIVE_RESULTS_ROOT / verifier / f'fold_{fold}' / 'training_summary.json'
        if not path.is_file():
            raise RuntimeError(f'missing completed fold: {path}')
        summary = json.loads(path.read_text(encoding='utf-8'))
        best = max(summary['history'], key=lambda row: (row['exact_match_accuracy'], -row['epoch']))
        fold_records.append({'fold': fold, **best})
    total = sum(record['samples'] for record in fold_records)
    comparison[verifier] = {
        'weighted_exact_match_accuracy': sum(record['samples'] * record['exact_match_accuracy'] for record in fold_records) / total,
        'weighted_lightweight_top8_recall': sum(record['samples'] * record['lightweight_top8_recall'] for record in fold_records) / total,
        'validation_samples': total,
        'folds': fold_records,
        'train_batch_size': training_config.batch_size,
        'micro_batch_size': training_config.micro_batch_size,
        'evaluation_batch_size': training_config.evaluation_batch_size,
        'resident_storage_mode': resident_store.storage_mode,
        'qwen_layer_indices': resident_store.qwen_layer_indices,
    }
selected_final_epochs = max(1, int(round(statistics.median(
    record['epoch'] for record in comparison[FINAL_VERIFIER]['folds']
))))
comparison['final_training'] = {
    'verifier': FINAL_VERIFIER,
    'selected_epochs': selected_final_epochs,
    'selection_rule': 'rounded median of five fold-best epochs',
    'samples': len(samples),
}
comparison_path = DRIVE_RESULTS_ROOT / 'transformer_ablation_summary.json'
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(comparison, indent=2))
print('saved:', comparison_path)


{
  "transformer": {
    "weighted_exact_match_accuracy": 0.3251179863660199,
    "weighted_lightweight_top8_recall": 0.7012060828526482,
    "validation_samples": 9535,
    "folds": [
      {
        "fold": 0,
        "epoch": 19,
        "train_loss": 0.3431313033809072,
        "samples": 1907,
        "exact_match_accuracy": 0.33036182485579446,
        "lightweight_top8_recall": 0.7021499737808076,
        "effective_evaluation_batch_size": 256
      },
      {
        "fold": 1,
        "epoch": 13,
        "train_loss": 0.7355659349651067,
        "samples": 1907,
        "exact_match_accuracy": 0.3434714210802307,
        "lightweight_top8_recall": 0.7110644992134242,
        "effective_evaluation_batch_size": 256
      },
      {
        "fold": 2,
        "epoch": 8,
        "train_loss": 1.478949770288395,
        "samples": 1909,
        "exact_match_accuracy": 0.3132530120481928,
        "lightweight_top8_recall": 0.6909376636982714,
        "effective_evaluation_batch_si

## 11. Train one final Transformer scorer on all 9,535 rows

The epoch count is the rounded median of the five Transformer fold-best epochs. With the completed folds `(19, 13, 8, 7, 8)`, this selects 8 epochs. The same int8 resident token store, logical batch 128, probed micro-batch, AMP, and GPU confirmation remain active. This produces one final model, not an ensemble.

In [ ]:
from dataclasses import replace

final_checkpoint_path = FINAL_OUTPUT_DIR / 'final.pt'
if TRAIN_FINAL_SCORER:
    final_complete = False
    if final_checkpoint_path.is_file():
        checkpoint = torch.load(final_checkpoint_path, map_location='cpu', weights_only=True)
        final_complete = (
            checkpoint['epoch'] == selected_final_epochs
            and checkpoint['scorer_config']['verifier'] == FINAL_VERIFIER
        )
        if not final_complete:
            print(
                'incomplete or stale final checkpoint; restarting full-data training:',
                f'epoch={checkpoint["epoch"]}/{selected_final_epochs}',
            )
        del checkpoint
    if final_complete:
        print('final scorer already completed:', final_checkpoint_path)
    else:
        seed_everything(SEED)
        final_scorer = make_scorer(FINAL_VERIFIER)
        final_config = replace(
            training_config,
            max_epochs=selected_final_epochs,
            patience=selected_final_epochs,
        )
        print(
            'final training:',
            f'verifier={FINAL_VERIFIER}',
            f'samples={len(prepared_samples)}',
            f'epochs={selected_final_epochs}',
            f'logical/micro={final_config.batch_size}/{final_config.micro_batch_size}',
            f'resident={resident_store.storage_mode}',
        )
        train_token_fusion_scorer(
            final_scorer, resident_store, prepared_samples, [],
            FINAL_OUTPUT_DIR, final_config,
        )
        del final_scorer
        gc.collect()
        torch.cuda.empty_cache()
assert final_checkpoint_path.is_file(), final_checkpoint_path


## 12. Build the test token cache for final-model submission

The competition rules prohibit model ensembling. This submission uses only `final_scorer/final.pt`, trained for the CV-selected 8 epochs on all 9,535 rows. Before test preparation, the training resident store and its GPU candidate plans are released.

In [4]:
import gc
import shutil
import tarfile

for name in ('resident_store', 'prepared_samples', 'fold0_train', 'worst_case_rows'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after releasing train store: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB')

from snu_ordering.data import load_samples

TEST_CSV = WORKDIR / 'snuaichallenge_data/test.csv'
TEST_IMAGE_ROOT = WORKDIR / 'snuaichallenge_data/test'
SAMPLE_SUBMISSION = WORKDIR / 'snuaichallenge_data/sample_submission.csv'
test_samples = load_samples(TEST_CSV, TEST_IMAGE_ROOT, validate_images=True)
assert len(test_samples) == 819 and all(sample.permutation is None for sample in test_samples)
assert TEST_CHECKPOINT_PATH.is_file(), TEST_CHECKPOINT_PATH

persisted_test_relative_paths = set()
test_cache_shards = sorted(DRIVE_TEST_TOKEN_SHARDS.glob('test_token_cache_*.tar'))
for shard in test_cache_shards:
    with tarfile.open(shard, 'r') as archive:
        members = [member for member in archive.getmembers() if member.isfile()]
        persisted_test_relative_paths.update(member.name for member in members)
        archive.extractall(LOCAL_TEST_TOKEN_CACHE, members=members, filter='data')
print('test samples/restored shards/files:', len(test_samples), len(test_cache_shards), len(persisted_test_relative_paths))


VRAM after releasing train store: 0.00 GiB
test samples/restored shards/files: 819 9 819


In [5]:
from snu_ordering.features import ConvNeXtTinyFeatureExtractor
from snu_ordering.qwen_representations import FrozenQwenRepresentationExtractor
from snu_ordering.token_fusion_representations import (
    cache_token_representations_batched,
    find_pending_token_representations,
)

test_pending, test_valid_paths = find_pending_token_representations(
    test_samples, LOCAL_TEST_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
test_next_shard_index = max([int(path.stem.split('_')[-1]) for path in test_cache_shards], default=-1) + 1
test_shard_buffer = []

def persist_test_saved(paths):
    global test_next_shard_index, test_shard_buffer
    test_shard_buffer.extend(paths)
    while len(test_shard_buffer) >= CACHE_SHARD_SIZE:
        selected, test_shard_buffer = test_shard_buffer[:CACHE_SHARD_SIZE], test_shard_buffer[CACHE_SHARD_SIZE:]
        local_tar = WORKDIR / f'test_token_cache_{test_next_shard_index:04d}.tar'
        final_tar = DRIVE_TEST_TOKEN_SHARDS / local_tar.name
        temporary = final_tar.with_suffix('.tar.tmp')
        with tarfile.open(local_tar, 'w') as archive:
            for path in selected:
                archive.add(path, arcname=path.relative_to(LOCAL_TEST_TOKEN_CACHE).as_posix())
        shutil.copy2(local_tar, temporary)
        temporary.replace(final_tar)
        local_tar.unlink()
        print('persisted test shard:', final_tar.name)
        test_next_shard_index += 1

unpersisted_test_valid = [
    path for path in test_valid_paths
    if path.relative_to(LOCAL_TEST_TOKEN_CACHE).as_posix() not in persisted_test_relative_paths
]
persist_test_saved(unpersisted_test_valid)
if test_pending:
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    convnext = ConvNeXtTinyFeatureExtractor(device='cuda')
    qwen = FrozenQwenRepresentationExtractor(device_map='auto', dtype=dtype)
    remaining = test_pending
    for feature_batch_size in FEATURE_BATCH_CANDIDATES:
        if not remaining:
            break
        try:
            summary = cache_token_representations_batched(
                remaining, convnext=convnext, qwen=qwen,
                qwen_layer_indices=QWEN_LAYER_INDICES, batch_size=feature_batch_size,
                image_workers=IMAGE_WORKERS, on_batch_saved=persist_test_saved,
            )
            print('test feature batch selected:', feature_batch_size, summary)
            remaining = []
        except torch.cuda.OutOfMemoryError:
            print('test feature batch OOM; retrying smaller:', feature_batch_size)
            gc.collect()
            torch.cuda.empty_cache()
            remaining, _ = find_pending_token_representations(
                test_samples, LOCAL_TEST_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
            )
    if remaining:
        raise RuntimeError('No safe test feature extraction batch was found')
    del qwen, convnext
    gc.collect()
    torch.cuda.empty_cache()

if test_shard_buffer:
    selected, test_shard_buffer = test_shard_buffer, []
    local_tar = WORKDIR / f'test_token_cache_{test_next_shard_index:04d}.tar'
    final_tar = DRIVE_TEST_TOKEN_SHARDS / local_tar.name
    with tarfile.open(local_tar, 'w') as archive:
        for path in selected:
            archive.add(path, arcname=path.relative_to(LOCAL_TEST_TOKEN_CACHE).as_posix())
    temporary = final_tar.with_suffix('.tar.tmp')
    shutil.copy2(local_tar, temporary)
    temporary.replace(final_tar)
    local_tar.unlink()

test_pending_after, test_valid_after = find_pending_token_representations(
    test_samples, LOCAL_TEST_TOKEN_CACHE, qwen_layer_indices=QWEN_LAYER_INDICES
)
assert not test_pending_after and len(test_valid_after) == 819


## 13. Put every test feature and candidate plan in VRAM, then write the CSV

The complete test token store is resident on the GPU in int8 form. Inference uses the already validated batch size 256; probing 819/768-sample batches can trigger an unrecoverable CUDA illegal-memory-access before an ordinary OOM is raised. The prediction loop still halves the batch automatically on a normal CUDA OOM.

In [6]:
import csv
from dataclasses import replace

from snu_ordering.data import parse_rank_label, write_submission
from snu_ordering.token_fusion_scorer import TokenFusionStructuredScorer
from snu_ordering.token_fusion_training import (
    TokenFusionTrainingConfig,
    predict_token_fusion_samples,
    prepare_resident_token_fusion_samples,
)

DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

test_resident_store, prepared_test_samples = prepare_resident_token_fusion_samples(
    test_samples, LOCAL_TEST_TOKEN_CACHE, DEVICE, top_h=TOP_H,
    storage_mode=RESIDENT_STORAGE,
)
assert all(item.target_index is None for item in prepared_test_samples)
checkpoint = torch.load(TEST_CHECKPOINT_PATH, map_location='cpu', weights_only=True)
test_scorer = TokenFusionStructuredScorer(**checkpoint['scorer_config']).to(DEVICE)
test_scorer.load_state_dict(checkpoint['model_state_dict'], strict=True)
test_config = TokenFusionTrainingConfig(**checkpoint['training_config'])
del checkpoint
gc.collect()

test_evaluation_batch_size = min(256, len(prepared_test_samples))
test_config = replace(test_config, evaluation_batch_size=test_evaluation_batch_size)
torch.cuda.reset_peak_memory_stats(DEVICE)
predictions = predict_token_fusion_samples(
    test_scorer, test_resident_store, prepared_test_samples, test_config
)
torch.cuda.synchronize(DEVICE)
test_peak_gib = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
write_submission(
    test_samples, predictions, SUBMISSION_PATH,
    sample_submission_path=SAMPLE_SUBMISSION,
)
with SUBMISSION_PATH.open('r', encoding='utf-8', newline='') as file:
    submission_rows = list(csv.DictReader(file))
assert len(submission_rows) == len(test_samples) == 819
assert [row['Id'] for row in submission_rows] == [sample.sample_id for sample in test_samples]
assert all(set(parse_rank_label(row['Answer'])) == {1, 2, 3, 4} for row in submission_rows)
print('resident storage:', test_resident_store.storage_mode)
print(f'resident token features: {test_resident_store.estimated_bytes / 1024**3:.2f} GiB')
print('inference batch:', test_evaluation_batch_size)
print(f'inference peak VRAM: {test_peak_gib:.2f} GiB')
print('submission rows/path:', len(submission_rows), SUBMISSION_PATH)


[token cache scan] 250/819
[token cache scan] 500/819
[token cache scan] 750/819
[token cache scan] 819/819
[resident prepare] 250/819
[resident prepare] 500/819
[resident prepare] 750/819
[resident prepare] 819/819
resident token store: mode=int8, features=2.20 GiB, qwen_layers=(-1,), max_image_tokens=256
resident storage: int8
resident token features: 2.20 GiB
inference batch: 256
inference peak VRAM: 47.53 GiB
submission rows/path: 819 /content/drive/MyDrive/snu ai/results/token_fusion_cross_attention_v1/submission_transformer_full_train.csv
